# 🐞 NB2 · Bug Hunt: 8 planted reproducibility bugs · Caza de 8 errores sembrados
**Research Methods & Scientific Integrity in AI · Session 5 · UNMSM Deep Tech Doctorate · Dr. Loveleen Gaur**  
*Métodos de Investigación e Integridad Científica en IA · Sesión 5*

**EN:** Activity 2, pairs, 15 min. The code cell below runs and prints an impressive accuracy. Run it **three times**. Then find all **eight** defects and write the fix for each in the table.  
**ES:** Actividad 2, parejas, 15 min. Ejecuta la celda **tres veces**. Encuentra los **ocho** defectos y escribe la corrección de cada uno.

Hint categories / Pistas: randomness (3) · environment (2) · data (1) · leakage (1) · reporting/logging (1)

> **🇪🇸 Traducción al español**
>
> # 🐞 NB2 · Caza de errores: 8 fallos de reproducibilidad sembrados
> **Métodos de Investigación e Integridad Científica en IA · Sesión 5 · Doctorado en Deep Tech UNMSM · Dra. Loveleen Gaur**
>
> Actividad 2, en parejas, 15 min. La celda de código de abajo se ejecuta y muestra una exactitud impresionante. Ejecútala **tres veces**. Luego encuentra los **ocho** defectos y escribe la corrección de cada uno en la tabla.
>
> Categorías de pistas: aleatoriedad (3) · entorno (2) · datos (1) · fuga de datos (1) · reporte/registro (1)

In [1]:
# ⚠️ This cell runs cleanly and prints a nice number. It is NOT reproducible.
# ES: ⚠️ Esta celda se ejecuta sin problemas e imprime un número bonito. NO es reproducible.
# Importa pandas y numpy
import pandas as pd, numpy as np                      # (versions? nobody wrote them down)
# Importa PyTorch aunque no se usa (importación innecesaria "por si acaso")
import torch                                          # imported "just in case"
# Función para dividir en entrenamiento y prueba
from sklearn.model_selection import train_test_split
# Escalador que estandariza las variables
from sklearn.preprocessing import StandardScaler
# Red neuronal multicapa para clasificación
from sklearn.neural_network import MLPClassifier
# Conjunto de datos de cáncer de mama
from sklearn.datasets import load_breast_cancer

# Activa la autoafinación de cuDNN: más rápido pero introduce no determinismo (error 8)
torch.backends.cudnn.benchmark = True                 # "makes it faster"

# Ruta absoluta que solo funciona en la laptop de Ana (error 2); aquí está comentada
# DATA_PATH = "/Users/ana/Desktop/tesis/datos_finales_v3_OK.csv"   # works only on Ana's laptop
# Carga los datos como DataFrame (sustituto del archivo de arriba, que no está versionado)
data = load_breast_cancer(as_frame=True).frame        # stand-in for the file above

# Separa las variables predictoras (todas menos "target")
X = data.drop(columns="target").values
# Separa la etiqueta a predecir
y = data["target"].values
# FUGA DE DATOS: el escalador aprende de TODAS las filas antes de dividir
X = StandardScaler().fit_transform(X)                  # scale everything first

# Mejor exactitud encontrada hasta ahora
best = 0
# Repite la división 5 veces y se queda con la mejor (cherry-picking)
for i in range(5):                                     # "try a few splits"
    # División 80/20 SIN semilla: cambia en cada ejecución
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
    # Entrena la red (64 y 32 neuronas) SIN semilla: los pesos iniciales cambian cada vez
    model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500).fit(X_tr, y_tr)
    # Exactitud en el conjunto de prueba
    acc = model.score(X_te, y_te)
    # Conserva solo el máximo
    best = max(best, acc)

# Imprime solo la mejor exactitud (resultado inflado, no registrado)
print(f"Accuracy: {best:.4f}")                         # report the best one

Accuracy: 0.9825


## ✅ Solution · Solución
| # | Bug | Fix |
|---|---|---|
| 1 | No random seed anywhere: splits and weights change every run | Set Python, NumPy (and framework) seeds; pass `random_state` to split and model |
| 2 | Hard-coded absolute path `/Users/ana/...` | Relative path inside the repo (`data/raw.csv`) tracked by DVC |
| 3 | No requirements file; versions unknown | `requirements.txt` with pinned versions; print versions in the notebook |
| 4 | Dataset not versioned ("datos_finales_v3_OK") | DVC-track the file; record its hash in the README |
| 5 | Split inside an unseeded loop and **best-of-5** reported (cherry-picking) | One pre-registered protocol; repeated CV; report all runs |
| 6 | `torch` imported with no pinned version (and not needed) | Remove unused imports; pin what remains |
| 7 | Metric printed, never logged | Log params, metrics, data hash and code commit (MLflow) |
| 8 | `cudnn.benchmark=True`: non-deterministic kernel autotuning | `cudnn.benchmark=False`, `cudnn.deterministic=True`, `use_deterministic_algorithms(True)` |
| ⭐ | Leakage: `StandardScaler` fitted on ALL rows before the split | Split first; put the scaler inside a `Pipeline` |

> **🇪🇸 Traducción al español**
>
> ## ✅ Solución
> | # | Error | Corrección |
> |---|---|---|
> | 1 | No hay semilla aleatoria en ningún lado: las divisiones y los pesos cambian en cada ejecución | Fijar las semillas de Python, NumPy (y del framework); pasar `random_state` a la división y al modelo |
> | 2 | Ruta absoluta escrita a mano `/Users/ana/...` | Ruta relativa dentro del repositorio (`data/raw.csv`) controlada con DVC |
> | 3 | No hay archivo de requisitos; versiones desconocidas | `requirements.txt` con versiones fijadas; imprimir las versiones en el notebook |
> | 4 | Conjunto de datos sin versionar ("datos_finales_v3_OK") | Controlar el archivo con DVC; registrar su hash en el README |
> | 5 | División dentro de un bucle sin semilla y se reporta el **mejor de 5** (cherry-picking) | Un único protocolo pre-registrado; validación cruzada repetida; reportar todas las ejecuciones |
> | 6 | `torch` importado sin versión fijada (y sin necesidad) | Eliminar importaciones no usadas; fijar las versiones de las que queden |
> | 7 | La métrica se imprime pero nunca se registra | Registrar parámetros, métricas, hash de datos y commit del código (MLflow) |
> | 8 | `cudnn.benchmark=True`: autoafinación de kernels no determinista | `cudnn.benchmark=False`, `cudnn.deterministic=True`, `use_deterministic_algorithms(True)` |
> | ⭐ | Fuga de datos: `StandardScaler` ajustado con TODAS las filas antes de dividir | Dividir primero; poner el escalador dentro de un `Pipeline` |

In [2]:
# Módulos estándar: sistema operativo, aleatoriedad, JSON, hash, sistema, plataforma y advertencias
import os, random, json, hashlib, sys, platform, warnings
# NumPy y scikit-learn (se necesita sklearn para registrar su versión)
import numpy as np, sklearn
# Carga el conjunto de datos de cáncer de mama
from sklearn.datasets import load_breast_cancer
# Validación cruzada estratificada repetida y su evaluación
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
# Encadena escalador + modelo en un solo objeto
from sklearn.pipeline import make_pipeline
# Escalador que estandariza las variables
from sklearn.preprocessing import StandardScaler
# Red neuronal multicapa para clasificación
from sklearn.neural_network import MLPClassifier
# Silencia las advertencias para una salida limpia
warnings.filterwarnings("ignore")

# Semilla global única
SEED = 42
# Fija las semillas de Python y NumPy
random.seed(SEED); np.random.seed(SEED)                       # (1) seeds
# Carga los datos (en un proyecto real: leer data/raw.csv versionado con DVC)
data = load_breast_cancer(as_frame=True).frame                 # (2)(4) in a real project: pd.read_csv("data/raw.csv") under DVC
# Calcula la huella MD5 de los datos para detectar cualquier cambio futuro
data_hash = hashlib.md5(data.to_csv(index=False).encode()).hexdigest()
# Separa variables predictoras (X) y etiqueta (y)
X, y = data.drop(columns="target").values, data["target"].values

# Pipeline: el escalador se ajusta dentro de cada partición de entrenamiento (sin fuga) y luego se entrena la red con semilla
model = make_pipeline(StandardScaler(),                        # (⭐) scaler fitted inside each training fold
                      # Segundo paso del Pipeline: red neuronal (64 y 32 neuronas) con semilla fija; cierra make_pipeline
                      MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=SEED))
# Validación cruzada 5 particiones x 5 repeticiones = 25 pruebas, con semilla (sin cherry-picking)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)   # (5) no cherry-picking
# Obtiene la exactitud de cada una de las 25 particiones
scores = cross_val_score(model, X, y, cv=cv)

# Registro del experimento: versiones, semilla, esquema de validación, huella de datos y resultados (media y DE)
record = {"python": platform.python_version(), "numpy": np.__version__, "sklearn": sklearn.__version__,   # (3)(6)
          # Continuación del registro: semilla, esquema de validación y huella MD5 de los datos
          "seed": SEED, "cv": "5x5 repeated stratified", "data_md5": data_hash,
          # Continuación del registro: exactitud media, desviación estándar y número de particiones; cierra el diccionario
          "accuracy_mean": round(scores.mean(), 4), "accuracy_sd": round(scores.std(ddof=1), 4), "n_folds": len(scores)}
# Imprime el registro en JSON legible (en NB4 esto se envía a MLflow)
print(json.dumps(record, indent=2))                            # (7) in NB4 this goes to MLflow

{
  "python": "3.13.18",
  "numpy": "2.1.3",
  "sklearn": "1.6.1",
  "seed": 42,
  "cv": "5x5 repeated stratified",
  "data_md5": "5afc23b9622f8f9ffa60824f4e97d0d5",
  "accuracy_mean": 0.975,
  "accuracy_sd": 0.0104,
  "n_folds": 25
}


**EN:** Compare: the buggy cell reports the *best* of five unseeded splits; the fixed cell reports mean ± SD over 25 folds, with versions and a data fingerprint. The honest number is lower and it is the one a stranger can reproduce.  
**ES:** La celda con errores reporta el *mejor* de cinco; la corregida reporta media ± DE sobre 25 particiones, con versiones y huella de datos.

> **🇪🇸 Traducción al español**
>
> Compara: la celda con errores reporta el *mejor* de cinco divisiones sin semilla; la celda corregida reporta media ± DE sobre 25 particiones, con versiones y una huella de los datos. El número honesto es más bajo y es el que un extraño puede reproducir.